In [1]:

import pandas as pd
from scipy.stats import spearmanr

from wearable_affect.data import DEFAULT_DATA_DIR, PROJECT_ROOT, list_subjects, load_self_reports
from wearable_affect.tracking import load_run_results

In [2]:
subjects = list_subjects(DEFAULT_DATA_DIR)
reports = pd.concat([load_self_reports(DEFAULT_DATA_DIR, s) for s in subjects])
wide = reports.pivot(index="subject_id", columns="condition", values=["valence", "arousal"])

check = pd.DataFrame({
    "arousal_base": wide["arousal"]["Base"],
    "arousal_tsst": wide["arousal"]["TSST"],
    "valence_base": wide["valence"]["Base"],
    "valence_tsst": wide["valence"]["TSST"],
}).reindex(subjects)
check["arousal_change"] = check["arousal_tsst"] - check["arousal_base"]
check["valence_change"] = check["valence_tsst"] - check["valence_base"]
check

,arousal_base,arousal_tsst,valence_base,valence_tsst,arousal_change,valence_change
subject_id,,,,,,
S10,2.0,8.0,6.0,3.0,6.0,-3.0
S11,2.0,6.0,6.0,4.0,4.0,-2.0
S13,3.0,9.0,5.0,4.0,6.0,-1.0
S14,2.0,7.0,5.0,3.0,5.0,-2.0
S15,5.0,8.0,6.0,4.0,3.0,-2.0
S16,2.0,8.0,7.0,3.0,6.0,-4.0
S17,2.0,9.0,7.0,1.0,7.0,-6.0
S2,2.0,4.0,7.0,5.0,2.0,-2.0
S3,4.0,7.0,7.0,7.0,3.0,0.0


In [ ]:
features = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "features.parquet")
scl = features.groupby(["subject_id", "target"])["scl_mean"].mean().unstack()
baseline = load_run_results("all_handcrafted-lightgbm")

check["scl_change"] = scl[1] - scl[0]
check["auroc"] = baseline["auroc"]

print(spearmanr(check["arousal_change"], check["auroc"]))
print(spearmanr(check["arousal_change"], check["scl_change"]))

SignificanceResult(statistic=np.float64(0.18350844283704135), pvalue=np.float64(0.5126851293374058))
SignificanceResult(statistic=np.float64(0.0924860534719655), pvalue=np.float64(0.7430437316103415))
